In [1]:
import pandas as pd
import feature_engineering_helper as hf
import pickle


In [ ]:

### change this to 1 for quick test, 0 for full run
quick_test = 1
### change this to 1 for quick test, 0 for full run


if quick_test == 1:
    n_features_to_select=40
    step=50
    print("Quick test mode")
else:
    n_features_to_select=1
    step=5
    print("Full run mode")



Quick test mode


In [3]:
data_prefix = '../0_data/processed_data/'
figure_prefix = '../Figures/'
non_feature_cols = ['SMILES', 'MP', 'Type', 'Ro5']
model_lists = ['RF', 'LGB', 'XGB']


if quick_test == 1:
    df_all_feature = pd.read_parquet(data_prefix + 'data_with_all_features_scaled.parquet').head(500)
else:
    df_all_feature = pd.read_parquet(data_prefix + 'data_with_all_features.parquet')

df_all_feature

,MACCS_0,MACCS_1,MACCS_10,MACCS_100,MACCS_101,MACCS_102,MACCS_103,MACCS_104,MACCS_105,MACCS_106,...,RDKit_fr_tetrazole,RDKit_fr_thiazole,RDKit_fr_thiocyan,RDKit_fr_thiophene,RDKit_fr_unbrch_alkane,RDKit_fr_urea,RDKit_qed,Ro5,SMILES,Type
0,0.0,0.0,0.0,-0.373111,-0.618592,2.068791,-0.441324,-0.430137,-0.639017,-0.480499,...,-0.055488,-0.127955,-0.024307,7.062219,-0.173756,-0.148915,-1.460638,1,ON=Cc1cscc1,Train
1,0.0,0.0,0.0,-0.373111,1.616573,-0.483374,-0.441324,2.324842,1.564905,-0.480499,...,-0.055488,-0.127955,-0.024307,-0.124610,-0.173756,-0.148915,0.947232,1,O=C1CC[C@]2(C(=C1)CC[C@@H]1[C@@H]2[C@H](O)C[C@...,Train
2,0.0,0.0,0.0,-0.373111,-0.618592,2.068791,-0.441324,-0.430137,-0.639017,-0.480499,...,-0.055488,-0.127955,-0.024307,-0.124610,-0.173756,-0.148915,-1.520308,1,[O-][n+]1ccccc1,Train
3,0.0,0.0,0.0,-0.373111,1.616573,-0.483374,-0.441324,2.324842,1.564905,-0.480499,...,-0.055488,-0.127955,-0.024307,-0.124610,-0.173756,-0.148915,-0.708981,1,OC1CCC2(C(C1)CC=C1C2CCC2(C1CCC2C(CCC(C(C)C)C)C...,Train
4,0.0,0.0,0.0,-0.373111,-0.618592,-0.483374,-0.441324,-0.430137,-0.639017,-0.480499,...,-0.055488,-0.127955,-0.024307,-0.124610,-0.173756,-0.148915,0.292369,1,CC(=O)c1ccc(cc1)Br,Train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,0.0,0.0,0.0,-0.373111,-0.618592,2.068791,-0.441324,2.324842,-0.639017,-0.480499,...,-0.055488,-0.127955,-0.024307,-0.124610,-0.173756,-0.148915,-1.796603,1,ONC(c1ccc(cc1)N)C/C(=N/O)/c1ccccc1,Train
496,0.0,0.0,0.0,-0.373111,-0.618592,-0.483374,-0.441324,-0.430137,-0.639017,-0.480499,...,-0.055488,-0.127955,-0.024307,-0.124610,-0.173756,-0.148915,0.298195,1,COc1cccc(c1)P(c1cccc(c1)OC)c1cccc(c1)OC,Train
497,0.0,0.0,0.0,-0.373111,-0.618592,-0.483374,-0.441324,-0.430137,-0.639017,-0.480499,...,-0.055488,-0.127955,-0.024307,-0.124610,7.585194,-0.148915,-2.570514,1,CCCCCCCCCCCCCCCCCCCCCC(=O)OC,Train
498,0.0,0.0,0.0,-0.373111,-0.618592,-0.483374,-0.441324,-0.430137,-0.639017,-0.480499,...,-0.055488,-0.127955,-0.024307,-0.124610,0.739061,-0.148915,-0.013056,1,CCCCCCc1ccc(cc1)c1ccccc1,Train


In [4]:
data_with_features_train = df_all_feature[df_all_feature['Type'] == 'Train']
print(data_with_features_train.shape)

data_with_features_train.describe()

(500, 379)


,MACCS_0,MACCS_1,MACCS_10,MACCS_100,MACCS_101,MACCS_102,MACCS_103,MACCS_104,MACCS_105,MACCS_106,...,RDKit_fr_sulfone,RDKit_fr_term_acetylene,RDKit_fr_tetrazole,RDKit_fr_thiazole,RDKit_fr_thiocyan,RDKit_fr_thiophene,RDKit_fr_unbrch_alkane,RDKit_fr_urea,RDKit_qed,Ro5
count,500.0,500.0,500.0,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,...,500.000000,500.000000,500.000000,500.000000,5.000000e+02,500.000000,500.000000,500.000000,500.000000,500.000000
mean,0.0,0.0,0.0,-0.025037,-0.019568,-0.003567,-0.013581,-0.027910,-0.017511,-0.039892,...,-0.011797,-0.021446,-0.019334,0.026282,-2.430697e-02,-0.023994,0.048058,-0.043712,-0.057684,0.976000
std,0.0,0.0,0.0,0.971338,0.990986,0.998161,0.988428,0.973774,0.992699,0.967692,...,0.916142,0.749790,0.808439,1.080740,6.945843e-18,0.845229,1.257984,1.014186,1.041289,0.153202
min,0.0,0.0,0.0,-0.373111,-0.618592,-0.483374,-0.441324,-0.430137,-0.639017,-0.480499,...,-0.103781,-0.068915,-0.055488,-0.127955,-2.430697e-02,-0.124610,-0.173756,-0.148915,-3.330353,0.000000
25%,0.0,0.0,0.0,-0.373111,-0.618592,-0.483374,-0.441324,-0.430137,-0.639017,-0.480499,...,-0.103781,-0.068915,-0.055488,-0.127955,-2.430697e-02,-0.124610,-0.173756,-0.148915,-0.635430,1.000000
50%,0.0,0.0,0.0,-0.373111,-0.618592,-0.483374,-0.441324,-0.430137,-0.639017,-0.480499,...,-0.103781,-0.068915,-0.055488,-0.127955,-2.430697e-02,-0.124610,-0.173756,-0.148915,0.060386,1.000000
75%,0.0,0.0,0.0,-0.373111,1.616573,-0.483374,-0.441324,-0.430137,1.564905,-0.480499,...,-0.103781,-0.068915,-0.055488,-0.127955,-2.430697e-02,-0.124610,-0.173756,-0.148915,0.678595,1.000000
max,0.0,0.0,0.0,2.680168,1.616573,2.068791,2.265911,2.324842,1.564905,2.081170,...,9.094584,11.798202,18.021759,7.583892,-2.430697e-02,7.062219,17.626188,13.001393,2.156380,1.000000


In [ ]:
def feature_engineering_workflow(model_type, df):

    data = df.copy()
    tolerance = 0.01

    # Extract all feature columns
    all_feature_cols = data.drop(columns=non_feature_cols, axis=1).columns.tolist()
    print(f"Total number of features: {len(all_feature_cols)}")
    print()

    # Reduce features by variance threshold
    variance_threshold = 0.01
    print(f'Variance threshold feature selection: variance_threshold={variance_threshold}')
    df_X_variance = hf.reduce_features_by_variance(data[all_feature_cols], variance_threshold=variance_threshold)
    print()


    print(f'RFE feature selection: model={model_type}, tolerance={tolerance}, n_features_to_select={n_features_to_select}, step={step}')
    # Reduce features by RFE
    RFE_results = hf.reduce_features_by_RFE(df_X_variance, data['MP'], model = model_type, tolerance = tolerance, n_features_to_select=n_features_to_select, step=step, metric='rmse', cv_strategy=None)
    print()

    # Plot RFE results
    hf.RFE_plot(RFE_results, tolerance , model_type,save_path = figure_prefix + f'RFE_plot_{model_type}.png')


    df_X_RFE = df_X_variance[RFE_results['best_features']]


    return df_X_variance, RFE_results, df_X_RFE

In [ ]:
feature_engineering_dict = {}

for model_type in model_lists:
    print(f"Running feature engineering workflow with {model_type} model")
    df_X_variance, RFE_results, df_X_RFE = feature_engineering_workflow(model_type, data_with_features_train)
    feature_engineering_dict[(model_type)] = RFE_results
    print()

# pickle save the feature_engineering_dict
with open('feature_engineering_dict.pkl', 'wb') as f:
    pickle.dump(feature_engineering_dict, f)

Running feature engineering workflow with RF model
Total number of features: 375
Original features: 375
Removed features: 24
Remaining features: 351

RFE feature selection: model=RF, tolerance=0.01, n_features_to_select=20, step=50


RFE Feature Selection:  14%|█▍        | 1/7 iteration

Iteration 0/7 | Features: 301 | RMSE: 49.0010 ± 3.7831 | Removed: [MACCS_11, MACCS_13, MACCS_14, MACCS_15, MACCS_16, MACCS_17, MACCS_18, MACCS_19, MACCS_20, MACCS_21, MACCS_22, MACCS_25, MACCS_27, MACCS_29, MACCS_31, MACCS_33, MACCS_34, MACCS_39, MACCS_40, MACCS_44, MACCS_51, MACCS_55, MACCS_58, MACCS_60, MACCS_67, MACCS_8, RDKit_EState_VSA11, RDKit_fr_Al_COO, RDKit_fr_HOCCN, RDKit_fr_N_O, RDKit_fr_aldehyde, RDKit_fr_amidine, RDKit_fr_azo, RDKit_fr_barbitur, RDKit_fr_epoxide, RDKit_fr_guanido, RDKit_fr_imide, RDKit_fr_isothiocyan, RDKit_fr_lactam, RDKit_fr_morpholine, RDKit_fr_nitroso, RDKit_fr_oxime, RDKit_fr_phos_acid, RDKit_fr_phos_ester, RDKit_fr_piperzine, RDKit_fr_quatN, RDKit_fr_sulfonamd, RDKit_fr_sulfone, RDKit_fr_tetrazole, RDKit_fr_thiophene]


# Get the dataset

In [ ]:
data_with_features = pd.read_parquet(data_prefix + 'data_with_all_features.parquet')
data_with_features_scaled = pd.read_parquet(data_prefix + 'data_with_all_features_scaled.parquet')

In [ ]:
# load the feature_engineering_dict
with open('feature_engineering_dict.pkl', 'rb') as f:
    feature_engineering_dict = pickle.load(f)

In [ ]:

for model_type in model_lists:
    best_features = feature_engineering_dict[(model_type)]['best_features']
    print(f"Best features for {model_type} model ({len(best_features)} features):")
    print(best_features)
    print()

    data_selected_features = data_with_features[non_feature_cols + best_features]
    data_selected_features_scaled = data_with_features_scaled[non_feature_cols + best_features]

    data_selected_features.to_parquet(data_prefix + f'data_with_selected_features_{model_type}.parquet')
    data_selected_features_scaled.to_parquet(data_prefix + f'data_with_selected_features_{model_type}_scaled.parquet')



Best features for LGB model (51 features):
['RDKit_VSA_EState4', 'RDKit_BertzCT', 'RDKit_VSA_EState6', 'RDKit_MinAbsEStateIndex', 'RDKit_TPSA', 'RDKit_FpDensityMorgan3', 'RDKit_BCUT2D_MRHI', 'RDKit_BCUT2D_MRLOW', 'RDKit_BalabanJ', 'RDKit_MaxPartialCharge', 'RDKit_EState_VSA3', 'RDKit_SlogP_VSA2', 'RDKit_SMR_VSA10', 'RDKit_FpDensityMorgan1', 'RDKit_VSA_EState5', 'RDKit_BCUT2D_MWLOW', 'RDKit_MolLogP', 'RDKit_qed', 'RDKit_VSA_EState2', 'RDKit_Kappa3', 'RDKit_NHOHCount', 'RDKit_PEOE_VSA8', 'RDKit_VSA_EState1', 'RDKit_EState_VSA4', 'RDKit_MinPartialCharge', 'RDKit_MaxAbsEStateIndex', 'RDKit_VSA_EState3', 'RDKit_HallKierAlpha', 'RDKit_BCUT2D_CHGHI', 'RDKit_FpDensityMorgan2', 'RDKit_EState_VSA2', 'RDKit_EState_VSA5', 'RDKit_BCUT2D_MWHI', 'RDKit_Kappa2', 'RDKit_MaxAbsPartialCharge', 'RDKit_SlogP_VSA4', 'RDKit_Kappa1', 'RDKit_FractionCSP3', 'RDKit_BCUT2D_CHGLO', 'RDKit_Chi4n', 'RDKit_EState_VSA6', 'RDKit_PEOE_VSA7', 'RDKit_MinEStateIndex', 'RDKit_BCUT2D_LOGPLOW', 'RDKit_NumRotatableBonds', 'RDK